In [11]:
import pandas as pd
import numpy as np

In [12]:
postings_db = pd.read_csv('../data/processed/postings_cleaned.csv')
# remove congress 86 from postings_db
postings_db = postings_db[postings_db['congress_number'] != 86]
postings_db.head()


,bioguide_id,chamber,congress_number,region_type,region_code,party,start_date,end_date
555,A000002,Representative,87,State,VA,Democrat,1961-01-03,1963-01-02
556,A000016,Representative,87,State,MS,Democrat,1961-01-03,1963-01-02
557,A000024,Representative,87,State,IN,Republican,1961-01-03,1963-01-02
558,A000052,Representative,87,State,NY,Democrat,1961-01-03,1963-01-02
559,A000054,Representative,87,State,NJ,Democrat,1961-01-03,1962-06-30


In [13]:
# set the datatype of start_date and end_date to datetime
postings_db['start_date'] = pd.to_datetime(postings_db['start_date'])
postings_db['end_date'] = pd.to_datetime(postings_db['end_date'])
postings_db.dtypes

bioguide_id                object
chamber                    object
congress_number             int64
region_type                object
region_code                object
party                      object
start_date         datetime64[ns]
end_date           datetime64[ns]
dtype: object

In [16]:
# Open profiles_db
profiles_db = pd.read_csv('../data/processed/profiles_cleaned.csv')
profiles_db['birth_date'] = pd.to_datetime(profiles_db['birth_date'])
profiles_db.dtypes

bioguide_id            object
first_name             object
last_name              object
birth_date     datetime64[ns]
dtype: object

In [14]:
# make a date range for congress sessions
# Earliest date in start_date column
# latest date in end_date column
start_date = postings_db['start_date'].min()
end_date = postings_db['end_date'].max()
date_range = pd.date_range(start=start_date, end=end_date, freq='D')
date_range

DatetimeIndex(['1961-01-03', '1961-01-04', '1961-01-05', '1961-01-06',
               '1961-01-07', '1961-01-08', '1961-01-09', '1961-01-10',
               '1961-01-11', '1961-01-12',
               ...
               '2024-12-25', '2024-12-26', '2024-12-27', '2024-12-28',
               '2024-12-29', '2024-12-30', '2024-12-31', '2025-01-01',
               '2025-01-02', '2025-01-03'],
              dtype='datetime64[ns]', length=23377, freq='D')

In [17]:
# define a function to make a list of active congress members on a given date
def get_active_members(date, postings_db):
    active_members = postings_db[(postings_db['start_date'] <= date) & (postings_db['end_date'] >= date)]
    return active_members['bioguide_id'].unique().tolist()

# define a function to calculate age given birth_date and a date
def calculate_age(birth_date, date):
    age = date.year - birth_date.year - ((date.month, date.day) < (birth_date.month, birth_date.day))
    return age

In [21]:
# loop through date_range and get active members on that date

# Define a date for testing
date_range = [pd.Timestamp('2000-01-01'), pd.Timestamp('2010-01-02')]

for date in date_range:
    ages = []
    active_members = get_active_members(date, postings_db)
    for member in active_members:
        # calculate age on that date based on date and profiles_db[birth_date]
        birth_date = profiles_db.loc[profiles_db['bioguide_id'] == member, 'birth_date']
        age = calculate_age(birth_date, date)
        # append to a list
        ages.append(age)
    # calculate average, min and maxage for list of ages
    average_age = np.mean(ages)
    min_age = np.min(ages)
    max_age = np.max(ages)
    print(f'On {date}, the average age of active congress members was {average_age}, with a minimum age of {min_age} and a maximum age of {max_age}')



AttributeError: 'Series' object has no attribute 'year'